In [92]:
import torch
import json
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

In [93]:
df = pd.read_csv('model/parkinsons.data')

In [94]:
with open('model/important_features.json', 'r') as f:
    important_features = json.load(f)

y = df['status']
X = df[important_features]
X.shape

(195, 7)

In [95]:
X_train, X_test, y_train, y_test = train_test_split(X,y, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X)
x_test = scaler.transform(X)        #in case test only transform, fit will cause leakage


In [96]:
X_test = X_test.to_numpy()  #since only transform returns df, converting to np.array manually

In [97]:
with open('model/scaler.pkl', 'wb') as f:
    scaler = pickle.dump(scaler, f)

In [98]:
X_tr_tensor = torch.from_numpy(X_train)
X_ts_tensor = torch.from_numpy(X_test)
y_tr_tensor = y_train.to_numpy(copy=True)
y_ts_tensor = y_test.to_numpy(copy=True)
y_tr_tensor = torch.from_numpy(y_tr_tensor)
y_ts_tensor = torch.from_numpy(y_ts_tensor)


In [99]:
print("Mean of labels:", y_tr_tensor.float().mean().item())

Mean of labels: 0.7534246444702148


Since the mean is 0.75 so the dataset is imbalanced, more 1 labelled. So we will modify the loss function. Oversampling and undersampling are good too. But best option is to adjust the loss function.

In [100]:
input_shape = X_tr_tensor.shape[1]
input_shape

7

In [106]:

class SimpleNN():
    def __init__(self, input_shape):
        self.weights1 = torch.randn(input_shape, 6, requires_grad = True, dtype = torch.float64)
        self.bias1 = torch.zeros(6, requires_grad = True, dtype = torch.float64)
        self.weights2 = torch.randn(6, 4, requires_grad = True, dtype = torch.float64)
        self.bias2 = torch.zeros(4, requires_grad = True, dtype = torch.float64)
        self.weights3 = torch.randn(4, 1, requires_grad = True, dtype = torch.float64)
        self.bias3 = torch.zeros(1, requires_grad = True, dtype = torch.float64)

    def forward(self, X):
        z1 = torch.matmul(X, self.weights1) + self.bias1
        a1 = torch.relu(z1)
        
        z2 = torch.matmul(a1, self.weights2) + self.bias2
        a2 = torch.relu(z2)

        z3 = torch.matmul(a2, self.weights3) + self.bias3
        y_pred = torch.sigmoid(z3)

        return y_pred

    def loss_func(self, y_pred, y):
        # epsilon = 1e-7
        # y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        # # Corrected binary cross-entropy loss
        # loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
        # return loss

        #new loss function
        
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)

        w0 = 2.0   # minority
        w1 = 0.67  # majority

        loss = -(w1 * y * torch.log(y_pred) + 
         w0 * (1 - y) * torch.log(1 - y_pred)).mean()

        return loss

In [107]:
model = SimpleNN(X_tr_tensor.shape[1])
lr = 0.01
EPOCHS = 100

In [108]:
for i in range(EPOCHS):
    y_pred = model.forward(X_tr_tensor)
    loss = model.loss_func(y_pred, y_tr_tensor)
    print(f"Epoch:{i+1} Loss: {loss.item():.6f}     y_pred_mean(): {y_pred.mean().item()}")

    loss.backward()

    with torch.no_grad():
        model.weights1 -= lr*model.weights1.grad
        model.bias1 -= lr*model.bias1.grad
        model.weights2 -= lr*model.weights2.grad
        model.bias2 -= lr*model.bias2.grad
        model.weights3 -= lr*model.weights3.grad
        model.bias3 -= lr*model.bias3.grad
    
    model.weights1.grad.zero_()
    model.bias1.grad.zero_()
    model.weights2.grad.zero_()
    model.bias2.grad.zero_()
    model.weights3.grad.zero_()
    model.bias3.grad.zero_()


Epoch:1 Loss: 0.887761     y_pred_mean(): 0.5560056769951361
Epoch:2 Loss: 0.875230     y_pred_mean(): 0.5535776073113757
Epoch:3 Loss: 0.863252     y_pred_mean(): 0.5511013200315649
Epoch:4 Loss: 0.851826     y_pred_mean(): 0.5485800769101665
Epoch:5 Loss: 0.840953     y_pred_mean(): 0.546018041356054
Epoch:6 Loss: 0.830632     y_pred_mean(): 0.5434203239529899
Epoch:7 Loss: 0.820859     y_pred_mean(): 0.5407929946659819
Epoch:8 Loss: 0.811632     y_pred_mean(): 0.538143062868688
Epoch:9 Loss: 0.802943     y_pred_mean(): 0.5354784942861491
Epoch:10 Loss: 0.794785     y_pred_mean(): 0.5328081019451684
Epoch:11 Loss: 0.787147     y_pred_mean(): 0.5301414279266036
Epoch:12 Loss: 0.780019     y_pred_mean(): 0.5274885905879598
Epoch:13 Loss: 0.773384     y_pred_mean(): 0.5248600987390719
Epoch:14 Loss: 0.767227     y_pred_mean(): 0.5222666396329744
Epoch:15 Loss: 0.761530     y_pred_mean(): 0.5197188469353197
Epoch:16 Loss: 0.756273     y_pred_mean(): 0.5172270672718597
Epoch:17 Loss: 0.75

In [109]:
model_test = SimpleNN(X_ts_tensor.shape[1])
with torch.no_grad():
    y_pred = model.forward(X_ts_tensor)
    loss = model.loss_func(y_pred, y_ts_tensor)
    print(f"Loss: {loss.item():.6f}")
    

Loss: 3.004532


### now time to save the model parameters


In [110]:
torch.save({
    "W1": model.weights1.detach(),
    "b1": model.bias1.detach(),
    "W2": model.weights2.detach(),
    "b2": model.bias2.detach(),
    "W3": model.weights3.detach(),
    "b3": model.bias3.detach(),
    "input_size": X.shape[1]
}, "model/model.pth")